<a href="https://colab.research.google.com/github/AnastasiaKovalenko232/iad-practices/blob/main/practices-7/7_3_Text_%D1%81lassification_with_sklearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text classification with sklearn

https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.datasets import fetch_20newsgroups
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix

In [2]:
all_train = fetch_20newsgroups(subset='train')

In [3]:
all_train.target_names

['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

In [4]:
 print("\n".join(all_train.data[0].split("\n")[:10]))

From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 


In [5]:
categories = ['rec.autos', 'rec.sport.baseball', 'sci.space']

In [6]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'), categories=categories)
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'), categories=categories)

In [7]:
newsgroups_train.data[0][:500]

"\nThe Centaur that is being built for T4 would be a better bet to integrate \nonto the Proton as the T4/Centaur is designed for the Extremely Harsh \nenvorinment of the T4 launch. It is also closer to 4 m in diameter. \n\nYou've hit on the real kicker, however. The Centaur is pressure stabilized. \nIt cannot hold up its own weight without pressure in the tanks. Additionally, \nthe pressure difference between the two tanks must be maintained to ~+/- 5 psi. \nThat is rather tight to be rocking and rolling"

## Vectorization with CountVectorizer

In [8]:
# example 1

n_features = 1000
count_vectorizer = CountVectorizer(max_df=0.95, min_df=0.05,
                                max_features=n_features,
                                stop_words='english')

train_count_vectorizer = count_vectorizer.fit_transform(newsgroups_train.data)
test_count_vectorizer = count_vectorizer.transform(newsgroups_test.data)

clf = LogisticRegression(random_state=0).fit(train_count_vectorizer, newsgroups_train.target)
predicted = clf.predict(test_count_vectorizer)

print(classification_report(newsgroups_test.target, predicted))

              precision    recall  f1-score   support

           0       0.69      0.60      0.64       396
           1       0.60      0.79      0.68       397
           2       0.70      0.57      0.63       394

    accuracy                           0.65      1187
   macro avg       0.66      0.65      0.65      1187
weighted avg       0.66      0.65      0.65      1187



In [9]:
# example 2

count_vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 2))

train_count_vectorizer = count_vectorizer.fit_transform(newsgroups_train.data)
test_count_vectorizer = count_vectorizer.transform(newsgroups_test.data)

clf = LogisticRegression(random_state=0).fit(train_count_vectorizer, newsgroups_train.target)
predicted = clf.predict(test_count_vectorizer)

print(classification_report(newsgroups_test.target, predicted))

              precision    recall  f1-score   support

           0       0.79      0.93      0.85       396
           1       0.88      0.87      0.87       397
           2       0.93      0.77      0.84       394

    accuracy                           0.86      1187
   macro avg       0.87      0.86      0.86      1187
weighted avg       0.87      0.86      0.86      1187



## Vectorization with TFIDF

In [10]:
tfidf_vectorizer = TfidfVectorizer(max_df=500, min_df=10)

tfidf_train = tfidf_vectorizer.fit_transform(newsgroups_train.data)
tfidf_test =  tfidf_vectorizer.transform(newsgroups_test.data)

clf = LogisticRegression().fit(tfidf_train, newsgroups_train.target)

predicted = clf.predict(tfidf_test)
print(classification_report(newsgroups_test.target, predicted))

              precision    recall  f1-score   support

           0       0.87      0.84      0.85       396
           1       0.84      0.90      0.87       397
           2       0.88      0.85      0.86       394

    accuracy                           0.86      1187
   macro avg       0.86      0.86      0.86      1187
weighted avg       0.86      0.86      0.86      1187



In [11]:
#version 2.0
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# === КРОК 1: Завантаження підмножини даних (4 теми для швидкості) ===
print("Завантажуємо текстовий датасет 20 Newsgroups з серверу Sklearn...")
categories = ['sci.med', 'sci.space', 'comp.graphics', 'talk.politics.guns']

newsgroups_train = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', categories=categories, remove=('headers', 'footers', 'quotes'))

print(f"👍 Дані успішно завантажено!")
print(f"Кількість документів для навчання: {len(newsgroups_train.data)}")
print(f"Кількість документів для тесту: {len(newsgroups_test.data)}\n")
print("="*60 + "\n")

# === КРОК 2: Метод А — Модель на основі Bag of Words (CountVectorizer) ===
print("1. Запуск класифікації за допомогою CountVectorizer...")

# Обмежуємо кількість ознак до 5000 найчастіших слів
count_vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X_train_counts = count_vectorizer.fit_transform(newsgroups_train.data)
X_test_counts = count_vectorizer.transform(newsgroups_test.data)

# Навчаємо логістичну регресію
clf_counts = LogisticRegression(max_iter=500, random_state=42)
clf_counts.fit(X_train_counts, newsgroups_train.target)
pred_counts = clf_counts.predict(X_test_counts)

print("\n[РЕЗУЛЬТАТИ: CountVectorizer]")
print(classification_report(newsgroups_test.target, pred_counts, target_names=categories))
print("="*60 + "\n")

# === КРОК 3: Метод Б — Модель на основі TF-IDF (TfidfVectorizer) ===
print("2. Запуск класифікації за допомогою TF-IDF Vectorizer...")

# Векторизація з урахуванням ваги слів (TF-IDF)
tfidf_vectorizer = TfidfVectorizer(max_df=500, min_df=5, max_features=5000, stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(newsgroups_train.data)
X_test_tfidf = tfidf_vectorizer.transform(newsgroups_test.data)

# Навчаємо логістичну регресію на вагах TF-IDF
clf_tfidf = LogisticRegression(max_iter=500, random_state=42)
clf_tfidf.fit(X_train_tfidf, newsgroups_train.target)
pred_tfidf = clf_tfidf.predict(X_test_tfidf)

print("\n[РЕЗУЛЬТАТИ: TF-IDF Vectorizer]")
print(classification_report(newsgroups_test.target, pred_tfidf, target_names=categories))

print("\n🚀 БЛОКНОТ 7.3 УСПІШНО ВИКОНАНО!")

Завантажуємо текстовий датасет 20 Newsgroups з серверу Sklearn...
👍 Дані успішно завантажено!
Кількість документів для навчання: 2317
Кількість документів для тесту: 1543


1. Запуск класифікації за допомогою CountVectorizer...

[РЕЗУЛЬТАТИ: CountVectorizer]
                    precision    recall  f1-score   support

           sci.med       0.82      0.87      0.84       389
         sci.space       0.80      0.73      0.76       396
     comp.graphics       0.71      0.79      0.75       394
talk.politics.guns       0.84      0.77      0.80       364

          accuracy                           0.79      1543
         macro avg       0.79      0.79      0.79      1543
      weighted avg       0.79      0.79      0.79      1543


2. Запуск класифікації за допомогою TF-IDF Vectorizer...

[РЕЗУЛЬТАТИ: TF-IDF Vectorizer]
                    precision    recall  f1-score   support

           sci.med       0.88      0.89      0.89       389
         sci.space       0.88      0.85      0